# Large scale Gauss-Newton

Let's try it on a ViT model with layers using the layer formulation. 

## Imports and data

In [11]:
from timeit import timeit

import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
from typing import Sequence
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import itertools
from jax.scipy.linalg import block_diag
import optax
from datasets import load_dataset

from jax import random
# Seeding for random operations
main_rng = random.PRNGKey(42)

# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


In [2]:
def get_datasets():
    """Load MNIST train and test datasets into memory."""
    # Load the MNIST dataset
    dataset = load_dataset('mnist')

    # Convert datasets to numpy arrays
    train_ds = {
        'image': jnp.array(dataset['train']['image']),
        'label': jnp.array(dataset['train']['label'])
    }
    test_ds = {
        'image': jnp.array(dataset['test']['image']),
        'label': jnp.array(dataset['test']['label'])
    }

    # Normalize the images
    train_ds['image'] = train_ds['image'].astype(jnp.float32) / 255.0
    test_ds['image'] = test_ds['image'].astype(jnp.float32) / 255.0

    # Reshape to mimic 3D
    train_ds['image'] = jnp.expand_dims(train_ds['image'], axis=-1)
    test_ds['image'] = jnp.expand_dims(test_ds['image'], axis=-1)


    return train_ds, test_ds

In [3]:
# Get datasets as dict of JAX arrays.
train_ds, test_ds = get_datasets()

## Define a ViT in Jax

Using https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/JAX/tutorial15/Vision_Transformer.html

In [4]:
def img_to_patch(x, patch_size, flatten_channels=True):
    """
    Inputs:
        x - torch.Tensor representing the image of shape [B, H, W, C]
        patch_size - Number of pixels per dimension of the patches (integer)
        flatten_channels - If True, the patches will be returned in a flattened format
                           as a feature vector instead of a image grid.
    """
    B, H, W, C = x.shape
    x = x.reshape(B, H//patch_size, patch_size, W//patch_size, patch_size, C)
    x = x.transpose(0, 1, 3, 2, 4, 5)    # [B, H', W', p_H, p_W, C]
    x = x.reshape(B, -1, *x.shape[3:])   # [B, H'*W', p_H, p_W, C]
    if flatten_channels:
        x = x.reshape(B, x.shape[1], -1) # [B, H'*W', p_H*p_W*C]
    return x

In [5]:
class AttentionBlock(nn.Module):
    embed_dim : int   # Dimensionality of input and attention feature vectors
    hidden_dim : int  # Dimensionality of hidden layer in feed-forward network
    num_heads : int   # Number of heads to use in the Multi-Head Attention block
    dropout_prob : float = 0.0  # Amount of dropout to apply in the feed-forward network

    def setup(self):
        self.attn = nn.MultiHeadDotProductAttention(num_heads=self.num_heads)
        self.linear = [
            nn.Dense(self.hidden_dim),
            nn.gelu,
            nn.Dropout(self.dropout_prob),
            nn.Dense(self.embed_dim)
        ]
        self.layer_norm_1 = nn.LayerNorm()
        self.layer_norm_2 = nn.LayerNorm()
        self.dropout = nn.Dropout(self.dropout_prob)

    def __call__(self, x, train=True):
        inp_x = self.layer_norm_1(x)
        attn_out = self.attn(inputs_q=inp_x, inputs_kv=inp_x)
        x = x + self.dropout(attn_out, deterministic=not train)

        linear_out = self.layer_norm_2(x)
        for l in self.linear:
            linear_out = l(linear_out) if not isinstance(l, nn.Dropout) else l(linear_out, deterministic=not train)
        x = x + self.dropout(linear_out, deterministic=not train)
        return x

In [23]:
class Preprocessor(nn.Module):
    embed_dim: int
    patch_size: int
    num_patches: int

    def setup(self):
        self.input_layer = nn.Dense(self.embed_dim)

    def __call__(self, x, cls_token, pos_embedding):
        # Preprocess input
        x = img_to_patch(x, self.patch_size)
        B, T, _ = x.shape
        x = self.input_layer(x)

        # Add CLS token and positional encoding
        cls_token = cls_token.repeat(B, axis=0)
        x = jnp.concatenate([cls_token, x], axis=1)
        x = x + pos_embedding[:, :T + 1]
        return x


class ClassificationHead(nn.Module):
    embed_dim: int
    num_classes: int

    def setup(self):
        self.mlp_head = nn.Sequential([
            nn.LayerNorm(),
            nn.Dense(self.num_classes)
        ])

    def __call__(self, x):
        # Perform classification prediction
        cls = x[:, 0]
        out = self.mlp_head(cls)
        return out


class VisionTransformer(nn.Module):
    embed_dim: int     # Dimensionality of input and attention feature vectors
    hidden_dim: int    # Dimensionality of hidden layer in feed-forward network
    num_heads: int     # Number of heads to use in the Multi-Head Attention block
    num_channels: int  # Number of channels of the input (3 for RGB)
    num_layers: int    # Number of layers to use in the Transformer
    num_classes: int   # Number of classes to predict
    patch_size: int    # Number of pixels that the patches have per dimension
    num_patches: int   # Maximum number of patches an image can have
    dropout_prob: float = 0.0  # Amount of dropout to apply in the feed-forward network

    def setup(self):
        # Layers/Networks
        self.transformer = [AttentionBlock(self.embed_dim,
                                           self.hidden_dim,
                                           self.num_heads,
                                           self.dropout_prob) for _ in range(self.num_layers)]
        self.dropout = nn.Dropout(self.dropout_prob)

        # Parameters/Embeddings
        self.cls_token = self.param('cls_token',
                                    nn.initializers.normal(stddev=1.0),
                                    (1, 1, self.embed_dim))
        self.pos_embedding = self.param('pos_embedding',
                                        nn.initializers.normal(stddev=1.0),
                                        (1, 1 + self.num_patches, self.embed_dim))

        # Submodules
        self.preprocessor = Preprocessor(self.embed_dim, self.patch_size, self.num_patches)
        self.classification_head = ClassificationHead(self.embed_dim, self.num_classes)

    def __call__(self, x, train=True):
        # Preprocess input
        x = self.preprocessor(x, self.cls_token, self.pos_embedding)

        # Apply Transformer
        x = self.dropout(x, deterministic=not train)
        for attn_block in self.transformer:
            x = attn_block(x, train=train)

        # Perform classification prediction
        out = self.classification_head(x)
        return out

In [27]:
main_rng, x_rng = random.split(main_rng)
# x = random.normal(x_rng, (1, 28, 28, 1))
x = train_ds['image'][0:1]

visntrans = VisionTransformer(embed_dim=128,
                              hidden_dim=512,
                              num_heads=4,
                              num_channels=1,
                              num_layers=4,
                              num_classes=10,
                              patch_size=4,
                              num_patches=49,
                              dropout_prob=0.1)

# Initialize parameters of the Vision Transformer with random key and inputs
main_rng, init_rng, dropout_init_rng = random.split(main_rng, 3)
params = visntrans.init({'params': init_rng, 'dropout': dropout_init_rng}, x, True)['params']
# Apply encoder block with parameters on the inputs
# Since dropout is stochastic, we need to pass a rng to the forward
main_rng, dropout_apply_rng = random.split(main_rng)
out = visntrans.apply({'params': params}, x, train=True, rngs={'dropout': dropout_apply_rng})
print('Out', out.shape)

# del visntrans, params

Out (1, 10)


In [25]:

total_params = sum(x.size for x in jax.tree_util.tree_leaves(params))

# Print the number of parameters
print(f"Number of parameters: {total_params}")

Number of parameters: 803338
